Allows import of utils from root directory

In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

In [2]:
import pandas as pd
price_df = pd.read_csv("data/prices_round_0_day_-2.csv" , sep = ';')
price_df.head(5)


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-2,0,EMERALDS,9992,11,9990,25,NaN,NaN,10008,11,10010,25,NaN,NaN,10000.0,0.0
1,-2,0,TOMATOES,4993,7,4992,17,NaN,NaN,5007,7,5008,17,NaN,NaN,5000.0,0.0
2,-2,100,TOMATOES,4998,5,4993,7,4992.0,16.0,5007,7,5008,16,NaN,NaN,5002.5,0.0
3,-2,100,EMERALDS,9992,15,9990,20,NaN,NaN,10008,15,10010,20,NaN,NaN,10000.0,0.0
4,-2,200,TOMATOES,4994,6,4993,20,NaN,NaN,5008,6,5009,20,NaN,NaN,5001.0,0.0


In [3]:
trade_df = pd.read_csv("data/trades_round_0_day_-2.csv" , sep = ';')
trade_df.head(5)

,timestamp,buyer,seller,symbol,currency,price,quantity
0,900,NaN,NaN,TOMATOES,XIRECS,5008.0,2
1,1700,NaN,NaN,TOMATOES,XIRECS,5006.0,3
2,4000,NaN,NaN,EMERALDS,XIRECS,10008.0,7
3,4100,NaN,NaN,TOMATOES,XIRECS,5002.0,3
4,5200,NaN,NaN,EMERALDS,XIRECS,9992.0,5


In [4]:
tomatoe_price_df = price_df[price_df['product']=='TOMATOES'].reset_index(drop=True)
tomatoe_trade_df = trade_df[trade_df['symbol']=='TOMATOES'].reset_index(drop=True)

In [5]:
from order_flow_analysis import detect_levels, level_coverage

detect_levels(tomatoe_price_df)
level_coverage(tomatoe_price_df)



Detected 3 order book levels

  Level       Bid %    Ask %     Use?
  ------------------------------------
  L1         100.0%   100.0%        ✅
  L2         100.0%   100.0%        ✅
  L3           3.6%     3.6%        ❌

  Recommended n_levels: 2


({1: (np.float64(100.0), np.float64(100.0), np.True_),
  2: (np.float64(100.0), np.float64(100.0), np.True_),
  3: (np.float64(3.62), np.float64(3.5900000000000003), np.False_)},
 2)

As order book level 1 and 2 are filled the most we will use micro price 2

In [6]:
from order_flow_analysis import mid_price, multi_level_micro_price
from stat_utils import compute_returns

tomatoe_price_df['mid_price'] = mid_price(tomatoe_price_df)
tomatoe_price_df['micro_price'] = multi_level_micro_price(tomatoe_price_df, 2)
returns_t = compute_returns(tomatoe_price_df['micro_price'])

In [7]:
from stat_utils import stationarity_panel

print(stationarity_panel(tomatoe_price_df["mid_price"]))
print(stationarity_panel(tomatoe_price_df["micro_price"]))
print(stationarity_panel(returns_t["R_t"]))
print(stationarity_panel(returns_t["r_t"]))

/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


{'adf_stat': np.float64(-2.396896267902117), 'adf_pvalue': np.float64(0.14261433108715277), 'adf_usedlag': 9, 'kpss_stat': np.float64(4.450300800126923), 'kpss_pvalue': np.float64(0.01), 'kpss_lags': 59}


/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


{'adf_stat': np.float64(-2.426072940681994), 'adf_pvalue': np.float64(0.13451231889733795), 'adf_usedlag': 6, 'kpss_stat': np.float64(4.452068785127703), 'kpss_pvalue': np.float64(0.01), 'kpss_lags': 59}


/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


{'adf_stat': np.float64(-45.77343740637742), 'adf_pvalue': 0.0, 'adf_usedlag': 5, 'kpss_stat': np.float64(0.04807374760940935), 'kpss_pvalue': np.float64(0.1), 'kpss_lags': 40}
{'adf_stat': np.float64(-45.773487346424545), 'adf_pvalue': 0.0, 'adf_usedlag': 5, 'kpss_stat': np.float64(0.048063231919539914), 'kpss_pvalue': np.float64(0.1), 'kpss_lags': 40}


/home/user/projects/imc_prosperity_tutorial/stat_utils.py:144: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kps = kpss(x, regression="c", nlags="auto")


In [8]:
from stat_utils import distribution_summary

print(distribution_summary(tomatoe_price_df["mid_price"]))
print(distribution_summary(tomatoe_price_df["micro_price"]))
print(distribution_summary(returns_t["R_t"]))
print(distribution_summary(returns_t["r_t"]))

{'mean': np.float64(5007.9485), 'variance': np.float64(105.85763351335135), 'skewness': np.float64(0.40905285304638955), 'kurtosis': np.float64(2.260433528777026), 'jb_stat': np.float64(506.8313713893811), 'jb_pvalue': np.float64(8.769323168607893e-111)}
{'mean': np.float64(5007.962201889111), 'variance': np.float64(105.44969636664516), 'skewness': np.float64(0.4097928077836448), 'kurtosis': np.float64(2.2548046356316944), 'jb_stat': np.float64(511.32256716813527), 'jb_pvalue': np.float64(9.283576128726446e-112)}
{'mean': np.float64(1.417423839553914e-07), 'variance': np.float64(2.3630134256262543e-08), 'skewness': np.float64(0.062019216017029546), 'kurtosis': np.float64(5.789075771222977), 'jb_stat': np.float64(3242.676961904101), 'jb_pvalue': np.float64(0.0)}
{'mean': np.float64(1.2992856601851133e-07), 'variance': np.float64(2.3629905143296342e-08), 'skewness': np.float64(0.06091619327159958), 'kurtosis': np.float64(5.788883317648425), 'jb_stat': np.float64(3242.0043484934954), 'jb_

As the price and returns are sationary and distrubution is non normal, the next step is to investiagate market making

In [9]:
price_graph = go.Figure()
price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['mid_price'], mode='lines', name='Mid Price'))
price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['micro_price'], mode='lines', name='Micro Price'))

price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['bid_price_1'], mode='lines', name='Best Bid Price'))
price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['bid_price_2'], mode='lines', name='Level 2 Bid Price'))
price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['bid_price_3'], mode='markers', name='Level 3 Bid Price'))

price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['ask_price_1'], mode='lines', name='Best Ask Price'))
price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['ask_price_2'], mode='lines', name='Level 2 Ask Price'))
price_graph.add_trace(go.Scatter(x=tomatoe_price_df.index, y=tomatoe_price_df['ask_price_3'], mode='markers', name='Level 3 Ask Price'))
price_graph.show()



Potential pattern identified above, when l3 order is put l1 orders spread narrows, mid point price follows this trend, but micro price and 2nd level orders seem to usually stay stable. Now need to verify this pattern and try and extract alpha

In [10]:
from order_flow_analysis import order_levels_pattern

order_levels_pattern(tomatoe_price_df)



Detected 3 order book levels

  Level       Bid %    Ask %     Use?
  ------------------------------------
  L1         100.0%   100.0%        ✅
  L2         100.0%   100.0%        ✅
  L3           3.6%     3.6%        ❌

  Recommended n_levels: 2
SECTION 1: L3 Coverage (deepest detected level)
  L3 bid only:  3.62%  (n=362)
  L3 ask only:  3.59%  (n=359)
  L3 both:      0.00%  (n=0)
  L3 either:    7.21%  (n=721)
  L3 absent:    92.79%  (n=9279)

SECTION 2: Spread Compression at L1 and L2 During L3 Events
  L1 spread (L3 either)                    present=    7.1221  absent=   13.5270  diff=   -6.4049  p=0.0000 ***
  L2 spread (L3 either)                    present=   14.7351  absent=   16.0223  diff=   -1.2872  p=0.0000 ***
  L1 spread (L3 bid only)                  present=    7.6657  absent=   13.5270  diff=   -5.8613  p=0.0000 ***
  L1 spread (L3 ask only)                  present=    6.5738  absent=   13.5270  diff=   -6.9532  p=0.0000 ***
  L1 spread (L3 both)                   

/home/user/projects/imc_prosperity_tutorial/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
/home/user/projects/imc_prosperity_tutorial/.venv/lib/python3.13/site-packages/statsmodels/tsa/stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


This confrims that L3 never appears on both sides simultaneously (0 co-occurrence), this is the most important finding. L3 is always one-sided, meaning it's a deliberate directional order. L1 moves aggressively toward mid, L2 doesn't move at all. This definitively confirms L2 is the true price anchor and L1 is just reacting to the L3 order presence.

This confirms that it is structural bot behaviour not just chance, next is too check for spoofing.

In [11]:
from order_flow_analysis import analyse_deep_level_spoofing, calculate_baseline_fill_rate, analyse_cross_level_spoofing

print(calculate_baseline_fill_rate(tomatoe_price_df, tomatoe_trade_df, 1))
print(calculate_baseline_fill_rate(tomatoe_price_df, tomatoe_trade_df, 2))
print(calculate_baseline_fill_rate(tomatoe_price_df, tomatoe_trade_df, 3))

analyse_deep_level_spoofing(tomatoe_price_df, tomatoe_trade_df)
analyse_cross_level_spoofing(tomatoe_price_df, tomatoe_trade_df)

{'total': {'fill_rate': 0.02649188743239527, 'withdrawn': 10909, 'executed': 289}, 'bid': {'fill_rate': 0.03320420586607637, 'withdrawn': 5421, 'executed': 180}, 'ask': {'fill_rate': 0.019861516034985423, 'withdrawn': 5488, 'executed': 109}}
{'total': {'fill_rate': 0.0, 'withdrawn': 21515, 'executed': 0}, 'bid': {'fill_rate': 0.0, 'withdrawn': 10622, 'executed': 0}, 'ask': {'fill_rate': 0.0, 'withdrawn': 10893, 'executed': 0}}
{'total': {'fill_rate': 0.0, 'withdrawn': 17, 'executed': 0}, 'bid': {'fill_rate': 0.0, 'withdrawn': 3, 'executed': 0}, 'ask': {'fill_rate': 0.0, 'withdrawn': 14, 'executed': 0}}
SECTION 4: Spoofing Analysis (L3) | Bid vs Ask Breakdown
Side          Withdrawn     Executed   Fill Ratio
--------------------------------------------------------------------------------
TOTAL                17            0        0.00%
BID                   3            0        0.00%
ASK                  14            0        0.00%
  VERDICT (BID): 🚨 SPOOFING
  VERDICT (ASK): 🚨 SPOOF

{'l1_bid_delta': 0.0408698682079977}

Though this sounds crazy it confirms the market is very heavily spoofed, which we can verify as the trade csv only has 600 lines, the cross level validation also tells, us that when l3 is spoofed, the number of l1 orders filled doubles, this tells us that the bot doing the spoofing does it when trying to fill there orders.